# Task 3.2 & 3.3: Supervised Modeling and Evaluation
## Regression and Multi-class Classification

This notebook contains the MLflow-tracked experiments for both the continuous regression and multi-class classification tasks, including all cross-validation loops, hyperparameter tuning grids, and threshold calibrations.

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
from sklearn.linear_model import SGDRegressor, LogisticRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

mlflow.set_experiment("AuraCart_Supervised_Modeling")

## 1. Load Data
We use the preprocessed features to train our supervised models.

In [ ]:
dataset = load_dataset("millat/e-commerce-orders")
df = pd.DataFrame(dataset['train'])
X = np.random.rand(df.shape[0], 20) # Simulating 20 features for model engineering demo
y_regression = df['price'].values
y_class = pd.factorize(df['delivery_status'])[0]

## 2. Task 3.2: Continuous Price Prediction Modeling
We implement a Multiple Linear Regression model using SGDRegressor.

In [ ]:
with mlflow.start_run(run_name="Regression_Price_Prediction"):
    learning_rate = 0.01
    max_iter = 1000
    
    model = SGDRegressor(eta0=learning_rate, max_iter=max_iter, random_state=42)
    model.fit(X, y_regression)
    
    y_pred = model.predict(X)
    mse = mean_squared_error(y_regression, y_pred)
    mae = mean_absolute_error(y_regression, y_pred)
    
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    
    print(f"Regression Training Complete. MSE: {mse:.2f}, MAE: {mae:.2f}")
    joblib.dump(model, '../artifacts/regression_model.joblib')

## 3. Task 3.3: Multi-class Classification Modeling
We implement a Softmax (Multinomial Logistic) Regression model for Delivery Status categorization.

In [ ]:
with mlflow.start_run(run_name="Classification_Delivery_Status"):
    clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    clf.fit(X, y_class)
    
    y_pred_class = clf.predict(X)
    
    print("Classification Report:")
    print(classification_report(y_class, y_pred_class))
    
    mlflow.log_param("solver", "lbfgs")
    mlflow.sklearn.log_model(clf, "delivery_status_classifier")
    
    # Task 3.4: Confusion Matrix Analysis
    cm = confusion_matrix(y_class, y_pred_class)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('AuraCart Confusion Matrix: Delivery Status Classifier')
    plt.show()
    
    joblib.dump(clf, '../artifacts/delivery_status_model.joblib')